In [1]:
import os, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
torch.cuda.empty_cache()

# pretrained model

In [2]:
from model_skingpt4 import *

/home/jq2uw/miniconda3/envs/skingpt4/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model, vis_processor, chat = ini t_chat(0, "skingpt_io_eval_llama2_13bchat")
sum(p.numel() for p in model.parameters() if p.requires_grad), sum(p.numel() for p in model.parameters())

Initializing Chat
Loading VIT


/home/jq2uw/miniconda3/envs/skingpt4/lib/python3.9/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading VIT Done
Loading Q-Former
Loading Q-Former Done
Loading LLM tokenizer
Loading LLM model


Loading checkpoint shards: 100%|███████████████████████████████████████████████████| 3/3 [00:03<00:00,  1.06s/it]


Loading LLM Done
Load 2 training prompts
Prompt Example 
###Human: <Img><ImageHere></Img> Could you describe the skin disease in this image for me? ###Assistant: 
Load BLIP2-LLM Checkpoint: /scratch/jq2uw/edit-skingpt4/model_skingpt4/weights/skingpt4_llama2_13bchat_base_pretrain_stage2.pth
Initialization Finished


(109099520, 14110861184)

In [4]:
print(type(model).__name__) 

skingpt_io


# data

In [5]:
# import os
# os.chdir("/scratch/jq2uw/edit-skingpt4")

In [6]:
from data_utils import *

In [7]:
df = process_tabular("./data")
train_df, val_df, test_df = split_df(df)
train_dataset = MIDASDataset(train_df, "./data")
val_dataset = MIDASDataset(val_df, "./data")
test_dataset = MIDASDataset(test_df, "./data")

id_patient: 733
id_filename: 3416
midas_path: 17
midas_path: {'malignant-bcc': 608, 'benign-melanocytic nevus': 578, 'nan': 497, 'benign-other': 421, 'benign-seborrheic keratosis': 242, 'malignant-melanoma': 238, 'malignant-scc': 203, 'malignant-ak': 191, 'malignant-sccis': 165, 'other-melanocytic lesion, possible re-excision (severe, spitz, aimp)': 109, 'benign-dermatofibroma': 51, 'other-non-neoplastic, inflammatory, infectious': 39, 'benign-hemangioma': 30, 'benign-fibrous papule': 18, 'malignant-other': 14, 'melanocytic tumor, possible re-excision (severe, spitz, aimp)': 6, 'unknown': 6}
y16_description: 16
y16: 16
y16: {'Basal Cell Carcinoma': 608, 'Melanocytic Nevus': 578, 'Other Benign': 421, 'Seborrheic Keratosis': 242, 'Melanoma': 238, 'Squamous Cell Carcinoma': 203, 'Actinic Keratosis': 191, 'Squamous Cell Carcinoma In Situ': 165, 'Melanocytic Lesion': 109, 'Dermatofibroma': 51, 'Non-neoplastic': 39, 'Hemangioma': 30, 'Fibrous Papule': 18, 'Other Malignant': 14, 'Melanocytic 

## finetune

In [8]:
from finetune_utils import *
# vis_processor is already initialized from run/init.py
TARGET = 'text_full'
train_ds_ft = MIDASFTSkGPTIODataset(train_dataset, vis_processor, TARGET)
val_ds_ft   = MIDASFTSkGPTIODataset(val_dataset,   vis_processor, TARGET)
train_loader = train_ds_ft.get_loader(batch_size=2, shuffle=True, num_workers=2)
val_loader   = val_ds_ft.get_loader(batch_size=2, shuffle=False, num_workers=2)


In [9]:
model.finetune(train_loader, val_loader, n_epochs=10, retrain=True,
        lr=1e-4, weight_decay=0.5, ckpt_path=f"./model_skingpt4/weights/finetune_skingpt_io_{TARGET}.pth")

epoch 1/10  train_loss=0.0115  val_loss=0.0063
epoch 2/10  train_loss=0.0040  val_loss=0.0041

KeyboardInterrupt: saved checkpoint to ./model_skingpt4/weights/finetune_skingpt_io_text_full.pth


## eval

In [ ]:
from eval_utils import *

In [10]:
# one example
i = 16
image = test_dataset[i]['image']
print(f"ground truth: {test_dataset[i]['y']['y3']}")
print("-" * 50)
print("Pretrained model")
model = load_model_weights(model, "./model_skingpt4/weights/skingpt4_llama2_13bchat_base_pretrain_stage2.pth")
resp = chat_with_image(chat, image, "Is the lesion malignant or benign, or unknown?", temperature=0.01)
print(resp)
print("-" * 50)
print(f"Finetuned model (y3)")
model = load_model_weights(model, "./model_skingpt4/weights/finetune_llama.pth")
resp = chat_with_image(chat, image, "Is the lesion malignant or benign, or unknown?", temperature=0.01)
print(resp)
print(f"Finetuned model ({TARGET})")
model = load_model_weights(model, f"./model_skingpt4/weights/finetune_skingpt4_{TARGET}.pth")
resp = chat_with_image(chat, image, "Is the lesion malignant or benign, or unknown?", temperature=0.01)
print(resp)
print("-" * 50)
print(f"Finetuned io model ({TARGET})")
model = load_model_weights(model, f"./model_skingpt4/weights/finetune_skingpt_io_{TARGET}.pth")
resp = chat_with_image(chat, image, "Is the lesion malignant or benign, or unknown?", temperature=0.01)
print(resp)

ground truth: malignant
--------------------------------------------------
Pretrained model
The image shows a lesion on the left side of the lung. It is difficult to determine if the lesion is malignant or benign without further information.
--------------------------------------------------
Finetuned model (y3)
malignant
Finetuned model (text_full)
The lesion type is benign. The patient is diagnosed with other benign - other benign skin lesions not specifically covered elsewhere. On pathology, verruca vulgaris. The patient is 71-year-old white male with Fitzpatrick type ii skin, no known allergies. The patient has no history of melanoma. The lesion is on the right forearm, imaged at dscope. It measures 1.0 × 1.0 cm. Clinical impressions include benign-other, benign-seborrheic keratosis, and malignant-SCC.
Finetuned model (text_full)
The lesion on the image is The lesion on the image is The lesion on the image is The lesion on the image is The lesion on the image is The lesion on the i

In [ ]:
# random 200 unique samples (reproducible)
import random
from torch.utils.data import Subset

random.seed(42)  # optional for determinism
idxs = random.sample(range(len(train_dataset)), k=min(200, len(train_dataset)))
subset = Subset(train_dataset, idxs)

res = eval_ft_skingpt4(chat, subset, 
                       temperature=0.01, target=TARGET, 
                       question="Is the lesion malignant or benign, or other?")
res

100%|████████████████████████████████████████████████████████████████████████| 200/200 [01:58<00:00,  1.68it/s]


{'accuracy': 0.695,
 'precision': 0.7017901593405774,
 'recall': 0.7089668615984405,
 'f1': 0.6776093931684458,
 'report': '              precision    recall  f1-score   support\n\n      benign       0.89      0.52      0.65        95\n   malignant       0.64      0.94      0.76        72\n       other       0.58      0.67      0.62        33\n\n    accuracy                           0.69       200\n   macro avg       0.70      0.71      0.68       200\nweighted avg       0.75      0.69      0.69       200\n',
 'confusion': array([[49, 33, 13],
        [ 1, 68,  3],
        [ 5,  6, 22]])}

In [ ]:
# random 200 unique samples (reproducible)
import random
from torch.utils.data import Subset

random.seed(42)  # optional for determinism
idxs = random.sample(range(len(test_dataset)), k=min(200, len(test_dataset)))
subset = Subset(test_dataset, idxs)

res = eval_ft_skingpt4(chat, subset, 
                       temperature=0.1, target=TARGET, 
                       question="Is the lesion malignant or benign, or other?")
res

100%|████████████████████████████████████████████████████████████████████████| 200/200 [01:25<00:00,  2.34it/s]


{'accuracy': 0.58,
 'precision': 0.4976360189118487,
 'recall': 0.5056140350877193,
 'f1': 0.49830268787730625,
 'report': '              precision    recall  f1-score   support\n\n      benign       0.37      0.28      0.32        50\n   malignant       0.71      0.74      0.72       114\n       other       0.42      0.50      0.46        36\n\n    accuracy                           0.58       200\n   macro avg       0.50      0.51      0.50       200\nweighted avg       0.57      0.58      0.57       200\n',
 'confusion': array([[14, 24, 12],
        [17, 84, 13],
        [ 7, 11, 18]])}

In [ ]:

# --- eval ---
from eval_utils import *
import os, torch

target = "y3"
question = "Is the lesion malignant or benign, or other?"
res_dir = f"./results/ft_skingpt4_{target}"
os.makedirs(res_dir, exist_ok=True)
for split_name, ds in [("test", test_dataset), ("train", train_dataset), ("val", val_dataset)]:
    res_fname = f"{res_dir}/eval_{split_name}.pth"  
    if os.path.exists(res_fname):
        continue
    res = eval_ft_skingpt4(chat, ds, temperature=0.01, target=target, question=question)
    torch.save(res, res_fname)

 68%|████████████████████████████████████████████████▋                       | 243/359 [02:19<01:14,  1.56it/s]

: 

In [ ]:
# --- eval ---
import os, torch
from pprint import pprint
target = "y3"
question = "Is the lesion malignant or benign, or other?"
res_dir = f"../results/ft_skingpt4_{target}"
for split_name in ["test", "train"]:
    print(split_name)
    res = torch.load(f"{res_dir}/eval_{split_name}.pth")
    pprint(res)


test
{'accuracy': 0.6120689655172413,
 'confusion': array([[ 73, 118,  42],
       [ 28, 289,  22],
       [ 26,  34,  64]]),
 'f1': 0.5515059015059015,
 'precision': 0.5767106492640801,
 'recall': 0.5606470426397919,
 'report': '              precision    recall  f1-score   support\n'
           '\n'
           '      benign       0.57      0.31      0.41       233\n'
           '   malignant       0.66      0.85      0.74       339\n'
           '       other       0.50      0.52      0.51       124\n'
           '\n'
           '    accuracy                           0.61       696\n'
           '   macro avg       0.58      0.56      0.55       696\n'
           'weighted avg       0.60      0.61      0.59       696\n'}
train
{'accuracy': 0.7576020851433536,
 'confusion': array([[519, 279, 130],
       [ 25, 877,  28],
       [ 19,  74, 348]]),
 'f1': 0.7469485594528837,
 'precision': 0.7742008041820251,
 'recall': 0.7620205926170888,
 'report': '              precision    recall  